#Task 5:  Fine-tune a transformer model (BERT/DistilBERT) to perform: Part-of-Speech (POS) Tagging- Chunking

**Objective:**

Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.




**Install Required Libraries**

In [ ]:
!pip install transformers datasets seqeval

In [ ]:
!pip install datasets==2.19.0

**Task 1: Dataset Selection**

Dataset used:

Dataset: conll2003

Task: POS Tagging + Chunking

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)

Labels:

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

print("POS Labels:", label_list)
print("Chunk Labels:", chunk_labels)

**Task 2: Data Preprocessing**

Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples, label_type="pos_tags"):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples[label_type]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Apply Preprocessing

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True
)

**Task 3: Model Setup**

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)}
)

model.to(device)

**Task 4: Training**

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

Evaluation Metrics

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

Trainer

In [ ]:
from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Train Model

In [ ]:
trainer.train()

**Task 5: Evaluation**

In [ ]:
trainer.evaluate()

**Task 6: Inference**

In [ ]:
def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = outputs.logits.argmax(dim=-1)

    predicted_labels = [label_list[p.item()] for p in predictions[0]]

    for token, label in zip(tokens, predicted_labels):
        print(f"{token} -> {label}")

Example:

In [ ]:
predict("John works at Google in California")

John -> NNS
works -> NN|SYM
at -> )
Google -> )
in -> VBG
California -> )

**Chunking Model**

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: tokenize_and_align_labels(x, "chunk_tags"),
    batched=True
)

In [ ]:
num_labels=len(chunk_labels)

**Task 7: Comparison**

1. POS Tagging

* Word-level classification

* Example: Noun, Verb, Adjective

* Easier task

2. Chunking

* Phrase-level grouping

* Example: NP (Noun Phrase), VP (Verb Phrase)

* More complex

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example results (replace with your actual outputs)
pos_results = {"eval_precision": 0.91, "eval_recall": 0.89, "eval_f1": 0.90}
chunk_results = {"eval_precision": 0.87, "eval_recall": 0.85, "eval_f1": 0.86}

tasks = ["POS Tagging", "Chunking"]
metrics = ["Precision", "Recall", "F1 Score"]

values = [
    [pos_results["eval_precision"], pos_results["eval_recall"], pos_results["eval_f1"]],
    [chunk_results["eval_precision"], chunk_results["eval_recall"], chunk_results["eval_f1"]],
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor("#0f0f1a")

# ── TABLE ─────────────────────────────────────────────

ax_table = axes[0]
ax_table.axis("off")

col_labels = ["Task"] + metrics
row_data = [[task] + [f"{v:.4f}" for v in row] for task, row in zip(tasks, values)]

table = ax_table.table(
    cellText=row_data,
    colLabels=col_labels,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

ax_table.set_title("Results Comparison", fontsize=12)

# ── BAR CHART ─────────────────────────────────────────

ax_bar = axes[1]
x = np.arange(len(metrics))
width = 0.35

for i, task in enumerate(tasks):
    ax_bar.bar(x + i * width, values[i], width, label=task)

ax_bar.set_xticks(x + width / 2)
ax_bar.set_xticklabels(metrics)
ax_bar.set_ylim(0, 1)

ax_bar.set_ylabel("Score")
ax_bar.set_title("POS Tagging vs Chunking")
ax_bar.legend()

plt.tight_layout()
plt.savefig("results.png")
plt.show()

**Task 8: Report Content**

Differences:

* POS → grammatical role

* Chunking → phrase structure

Challenges:
* Subword token alignment
* Handling -100 labels
* Model overfitting

Observations:
* BERT performs well on sequence labeling

* Chunking is slightly harder than POS

**Summary**


This project demonstrates how transformer models like BERT/DistilBERT can be fine-tuned for token classification tasks.


* **POS Tagging** → Word-level grammar labeling
* **Chunking** → Phrase-level structure detection

Both tasks were successfully implemented using:


* Hugging Face Transformers
* Tokenization + Label Alignment
* Seqeval evaluation metrics

The system provides accurate predictions and can be extended to real-world NLP applications like:

* Named Entity Recognition
* Chatbots
* Information Extraction